**HEALTH ECONOMICS & CLINICAL OUTCOMES**

# Personnes traitées pour hypertension : profil, recours aux soins et coûts

Les coûts de la santé à travers les données, atelier du 28.10.2026. David De Ridder, PhD.

14HS031, Master en sciences biomédicales, Université de Genève, 2026–2027.

**Version R.** Une version Python identique existe : `tp_hypertension.ipynb`.

## La question

Qui sont les personnes traitées pour hypertension chez un assureur-maladie, comment recourent-elles aux soins, et combien coûtent-elles par rapport aux autres assuré·es ?

On commence par découvrir les données, puis on regarde qui coûte et pourquoi. L'hypertension vient ensuite, en section 4.

## Les données

Les décomptes 2024 de 15 000 adultes assurés chez un assureur de Suisse romande. Elles sont **entièrement synthétiques** : elles ressemblent à de vraies données, mais ne concernent personne. Le fichier des médicaments ne contient qu'une partie des substances. Les variables sont décrites dans `dictionnaire_donnees.md`, à ouvrir seulement en cas de besoin.

## À rendre

Une note d'une page pour un·e responsable de santé publique : la population concernée, les traitements et le recours aux soins, les coûts, deux ou trois messages et une limite d'interprétation (section 5).

## Mode d'emploi

- Exécuter les cellules dans l'ordre, avec Maj + Entrée. Inutile de comprendre tout le code : les lignes qui commencent par `#` disent ce que fait chaque cellule.
- Dans Colab, le notebook s'ouvre avec R. En cas de doute : **Exécution → Modifier le type d'exécution**.
- **Exercices.** Sélectionner les lignes de la cellule, les décommenter (Ctrl + / ou Cmd + /), puis remplacer chaque `___`. Si le raccourci ne marche pas, par exemple avec certains claviers suisses, effacer à la main le `# ` au début de chaque ligne. Un indice se trouve sous chaque exercice.
- **Pièges d'interprétation** (A à D). Un graphique semble mener à une conclusion évidente. Dire pourquoi les données ne permettent pas de la tirer, puis exécuter la vérification.
- Les réponses se notent dans les cellules « Réponses du groupe ». Les exercices facultatifs sont pour les groupes qui avancent vite.

| Section | Contenu | Exercices | Pièges | Durée indicative |
|---|---|---|---|---|
| 1 | Prise en main et qualité des données | 1 | | 20 min |
| 2 | Qui sont les assuré·es ? | 2 (3 facultatif) | | 20 min |
| 3 | Qui coûte, et pourquoi ? | 4 (5 et 6 facultatifs) | A, B | 45 min |
| 4 | Personnes traitées pour hypertension : identification, profil, traitements, recours aux soins, coûts | 7, 9, 10 (8 facultatif) | C, D | 75 min |
| 5 | Synthèse et rédaction de la note | | | 20 min |

<details><summary><b>Rappels : les fonctions du tidyverse utilisées</b></summary>

| Fonction | Rôle |
|---|---|
| `filter()` | garder certaines lignes |
| `select()` | garder certaines colonnes |
| `mutate()` | créer ou modifier une variable |
| `group_by()` puis `summarise()` | résumer par groupe |
| `count()` | compter les lignes par valeur |
| `left_join()`, `inner_join()` | joindre deux tables par une clé commune |
| `pivot_wider()`, `pivot_longer()` | passer d'un format long à un format large, et inversement |
| `%>%` | enchaîner les opérations : `x %>% f()` équivaut à `f(x)` |

Dans ces fonctions, les noms de colonnes s'écrivent sans guillemets : `filter(assures, sexe == "F")`. Les graphiques utilisent `ggplot2`.

</details>

In [ ]:
# @title Préparation de l'environnement (à exécuter sans modification)
paquets <- c("dplyr", "tidyr", "readr", "ggplot2", "scales", "stringr", "forcats", "tibble", "purrr")
a_installer <- function() paquets[!vapply(paquets, requireNamespace, logical(1), quietly = TRUE)]
if (length(a_installer()) > 0) {
  # Sous Linux (Colab), paquets précompilés de Posit : quelques secondes au lieu de plusieurs minutes de compilation
  os <- if (file.exists("/etc/os-release")) readLines("/etc/os-release") else character(0)
  version_linux <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(version_linux) == 1) {
    options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                                    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])))
    try(install.packages(a_installer(), repos = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", version_linux)))
  }
  if (length(a_installer()) > 0) install.packages(a_installer(), repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(for (p in paquets) library(p, character.only = TRUE))

options(repr.plot.width = 8, repr.plot.height = 4.5, repr.plot.res = 120,   # taille des graphiques dans Colab
        width = 120, dplyr.summarise.inform = FALSE, readr.show_col_types = FALSE)
theme_set(theme_minimal(base_size = 13) +
            theme(plot.title = element_text(face = "bold"), panel.grid.minor = element_blank()))

ANNEE <- 2024
GROUPES <- c("HTA traitée", "Autres assuré·es")
COULEURS <- c("HTA traitée" = "#C0392B", "Autres assuré·es" = "#7F8C8D")   # un groupe mis en évidence, le reste en gris
COULEURS_SEXE <- c(F = "#EB6834", M = "#2A78D6")
NOMS_SEXE <- c(F = "Femmes", M = "Hommes")
COULEURS_CAUSE <- c(maladie = "#2A78D6", maternite = "#EB6834", accident = "#1BAF7A")
CATEGORIELLES <- c("#2A78D6", "#EB6834", "#1BAF7A", "#EDA100", "#E87BA4", "#008300", "#6250D6", "#E34948")
RAMPE <- c("#86B6EF", "#3987E5", "#1C5CAB", "#0D366B")                     # catégories ordonnées, du clair au foncé
CLASSES_OFSP <- c(19, 26, 36, 46, 56, 66, 76, 86, 120)                    # classes d'âge de la statistique fédérale
LIBELLES_OFSP <- c("19-25", "26-35", "36-45", "46-55", "56-65", "66-75", "76-85", "86+")

LIBELLES <- c(   # libellés des codes affichés dans les graphiques
  standard = "Standard", medecin_de_famille = "Médecin de famille", hmo = "HMO", telmed = "Télémédecine",
  medecin_generaliste = "Généraliste", medecin_specialiste = "Spécialiste", ambulatoire_hospitalier = "Ambulatoire hospitalier",
  physiotherapie = "Physiothérapie", gastro_enterologie = "Gastro-entérologie", gynecologie = "Gynécologie",
  orthopedie = "Orthopédie", orl = "ORL", franchise_chf = "Franchise (CHF)", modele_assurance = "Modèle d'assurance")

libelle <- function(code) {
  # Libellé lisible d'un code
  code <- as.character(code)
  unname(ifelse(code %in% names(LIBELLES), LIBELLES[code], str_to_sentence(str_replace_all(code, "_", " "))))
}

chf <- function(x) format(round(x), big.mark = "'", scientific = FALSE, trim = TRUE)   # montant avec séparateur de milliers
pct <- function(x, decimales = 0) percent(x, accuracy = 10^-decimales)                 # proportion en pourcentage
axe_chf <- label_number(big.mark = "'")
axe_pct <- label_percent()

cat("Environnement prêt.\n")

In [ ]:
# @title Chargement des données
# Enseignant·e : adresse du dossier contenant les fichiers CSV (ex. dépôt GitHub, lien « raw »)
URL_DONNEES <- "https://raw.githubusercontent.com/daderidd/heco-tp-hypertension/main/donnees/"

lire <- function(nom) {
  local <- file.path("donnees", nom)                     # dossier « donnees » déposé dans Colab
  read_csv(if (file.exists(local)) local else paste0(URL_DONNEES, nom), guess_max = 1e6, show_col_types = FALSE)
}

assures <- lire("assures.csv")
prestations <- lire("prestations.csv")
medicaments <- lire("medicaments.csv")
referentiel <- lire("referentiel_atc.csv")
prestataires <- lire("referentiel_prestataires.csv")

cat(sprintf("assures      : %8s lignes\n", chf(nrow(assures))))
cat(sprintf("prestations  : %8s lignes\n", chf(nrow(prestations))))
cat(sprintf("medicaments  : %8s lignes\n", chf(nrow(medicaments))))

---
## 1. Prise en main et qualité des données

Trois tables, reliées par l'identifiant de la personne, `id_assure` :

```
assures       une ligne par personne assurée
prestations   une ligne par poste de facture (consultation, analyse, séjour, etc.)
medicaments   une ligne par médicament délivré
```

Deux petites tables de référence complètent l'ensemble : les codes ATC des médicaments et la liste des prestataires.

In [ ]:
# Les 5 premières lignes de chaque table
head(assures, 5)
head(prestations, 5)
head(medicaments, 5)

**Exercice 1. Combien d'assuré·es par canton ?** *(≈ 3 min)*

Compter les assuré·es de chaque canton. Il suffit de remplacer `___` par le nom de la colonne du canton.

In [ ]:
## Exercice 1 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# count(assures, ___, sort = TRUE)

<details><summary><b>Indice</b></summary>

Le nom de la colonne se lit dans les premières lignes affichées ci-dessus. Dans `count()`, il s'écrit sans guillemets.

</details>

**Questions (exercice 1)**
1. Quel canton compte le plus d'assuré·es ? Lequel le moins ?
2. Combien de personnes n'ont pas de canton (`NA`) ?

Comme toutes les données de facturation, celles-ci contiennent des erreurs : âges impossibles, doublons, montants négatifs (des annulations), dates hors de l'année, codes ATC mal écrits. La cellule suivante les compte.

In [ ]:
assures <- assures %>% mutate(age = ANNEE - annee_naissance)          # âge au 31 décembre
colonnes_prest <- setdiff(names(prestations), "id_ligne")             # tout sauf l'identifiant de ligne
colonnes_med <- setdiff(names(medicaments), "id_ligne")

controles <- tibble(
  controle = c("Âge > 105 ans", "Sexe manquant", "Canton manquant", "Prestations : lignes en double",
               "Prestations : montant négatif", "Prestations : montant nul", paste("Prestations : date hors", ANNEE),
               "Médicaments : lignes en double", "Médicaments : code ATC mal écrit"),
  nombre = c(sum(assures$age > 105), sum(is.na(assures$sexe)), sum(is.na(assures$canton)),
             sum(duplicated(prestations[colonnes_prest])), sum(prestations$cout_chf < 0), sum(prestations$cout_chf == 0),
             sum(format(prestations$date_prestation, "%Y") != ANNEE), sum(duplicated(medicaments[colonnes_med])),
             sum(medicaments$code_atc != str_to_upper(str_trim(medicaments$code_atc)))))
controles

On corrige ces erreurs dans des copies des tables (`assures_ok`, `prest_ok`, `med_ok`), utilisées dans toute la suite. La cellule suivante s'exécute sans modification (double-cliquer dessus pour voir le code).

Une visite chez le médecin produit souvent plusieurs lignes : la consultation, une analyse de laboratoire, etc. Pour compter les recours aux soins, on regroupe les lignes d'une même personne, le même jour, pour un même type de prestation. C'est un **contact**.

In [ ]:
# @title Préparation des données (à exécuter sans modification)
# Assuré·es
assures_ok <- assures %>%
  mutate(age = if_else(age > 105, NA_real_, age),
         classe_age = cut(age, breaks = c(18, 39, 49, 59, 69, 79, 110),
                          labels = c("19-39", "40-49", "50-59", "60-69", "70-79", "80+")),
         classe_ofsp = cut(age, breaks = CLASSES_OFSP, right = FALSE, labels = LIBELLES_OFSP))

# Médicaments
med_ok <- medicaments[!duplicated(medicaments[colonnes_med]), ] %>%
  mutate(code_atc = str_to_upper(str_trim(code_atc)))

# Prestations
prest_ok <- prestations[!duplicated(prestations[colonnes_prest]), ] %>%
  filter(format(date_prestation, "%Y") == ANNEE)

# Contacts : une personne, un jour, un type de prestation, une cause, montant net positif
contacts <- prest_ok %>%
  group_by(id_assure, date_prestation, type_prestation, cause) %>%
  summarise(cout_chf = sum(cout_chf), specialite = first(na.omit(specialite)), .groups = "drop") %>%
  filter(cout_chf > 0)

cat(sprintf("Prestations : %s lignes → %s lignes nettoyées → %s contacts\n",
            chf(nrow(prestations)), chf(nrow(prest_ok)), chf(nrow(contacts))))
cat(sprintf("Médicaments : %s lignes → %s délivrances\n", chf(nrow(medicaments)), chf(nrow(med_ok))))

**Questions**
1. Dans la table `prestations`, une ligne correspond-elle à une visite chez le médecin ?
2. Quelle information manque pour savoir qui a de l'hypertension ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 2. Qui sont les assuré·es ?

L'âge, le sexe et les choix d'assurance pèsent lourd sur les dépenses. On les décrit avant toute comparaison.

In [ ]:
# Pyramide des âges par classes de 5 ans (la première va de 19 à 24 ans)
bornes <- c(19, seq(25, 95, 5), 120)
noms <- c(paste0(head(bornes, -2), "-", bornes[2:(length(bornes) - 1)] - 1), "95+")
pyramide <- assures_ok %>%
  filter(!is.na(age), !is.na(sexe)) %>%
  mutate(classe = cut(age, bornes, right = FALSE, labels = noms)) %>%
  count(classe, sexe) %>%
  complete(classe, sexe, fill = list(n = 0L))

options(repr.plot.width = 7, repr.plot.height = 5)
ggplot(pyramide, aes(x = if_else(sexe == "M", -n, n), y = classe, fill = sexe)) +
  geom_col(width = 0.85) +
  geom_vline(xintercept = 0, linewidth = 0.3) +
  scale_fill_manual(values = COULEURS_SEXE, labels = NOMS_SEXE, name = NULL) +
  scale_x_continuous(labels = function(v) chf(abs(v))) +
  labs(title = "Pyramide des âges des assuré·es", x = "Nombre de personnes", y = "Âge (ans)")

In [ ]:
# Franchise choisie selon l'âge (franchises intermédiaires regroupées)
GROUPES_FRANCHISE <- c(`300` = "300", `500` = "500", `1000` = "1000-2000", `1500` = "1000-2000", `2000` = "1000-2000", `2500` = "2500")
franchise <- assures_ok %>%
  filter(!is.na(classe_ofsp)) %>%
  mutate(franchise = factor(GROUPES_FRANCHISE[as.character(franchise_chf)], levels = c("300", "500", "1000-2000", "2500"))) %>%
  count(classe_ofsp, franchise) %>%
  group_by(classe_ofsp) %>% mutate(part = n / sum(n)) %>% ungroup()

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(franchise, aes(x = part, y = fct_rev(classe_ofsp), fill = franchise)) +
  geom_col(width = 0.8, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  scale_fill_manual(values = RAMPE, name = "Franchise (CHF)") +
  scale_x_continuous(labels = axe_pct) +
  labs(title = "Franchise choisie selon l'âge", x = "Part des assuré·es", y = "Classe d'âge")

**Exercice 2. Franchise maximale selon le sexe** *(≈ 7 min)*

Les femmes choisissent-elles moins souvent la franchise maximale (2 500 CHF) que les hommes ? Calculer, par classe d'âge, la part des femmes et des hommes qui l'ont choisie.

Pour comparer, dans toute la Suisse (OFSP, EFIND 2024) : 68 % des hommes et 54 % des femmes de 26 à 35 ans, 34 % et 25 % de 56 à 65 ans.

In [ ]:
## Exercice 2 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# part_2500 <- assures_ok %>%
#   group_by(classe_ofsp, ___) %>%
#   summarise(pct_2500 = 100 * mean(franchise_chf == ___), .groups = "drop") %>%
#   drop_na() %>%
#   pivot_wider(names_from = ___, values_from = pct_2500)
# part_2500

<details><summary><b>Indice</b></summary>

La comparaison `==` donne `TRUE` ou `FALSE` pour chaque personne. La moyenne de ces valeurs dans un groupe est une proportion. Le second critère de regroupement est le sexe : reprendre le nom de la colonne correspondante, sans guillemets.

</details>

**Questions (exercice 2)**
1. À âge égal, les femmes choisissent-elles moins souvent la franchise maximale que les hommes ? À quels âges l'écart est-il le plus grand ?
2. Quelles explications proposer ?

In [ ]:
# Modèle d'assurance selon le canton
MODELES <- c("standard", "medecin_de_famille", "hmo", "telmed")
modeles <- assures_ok %>%
  filter(!is.na(canton)) %>%
  count(canton, modele_assurance) %>%
  group_by(canton) %>% mutate(part = n / sum(n)) %>% ungroup() %>%
  mutate(modele = factor(libelle(modele_assurance), levels = libelle(MODELES)))

options(repr.plot.width = 8, repr.plot.height = 3.5)
ggplot(modeles, aes(x = part, y = fct_rev(canton), fill = modele)) +
  geom_col(width = 0.8, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  scale_fill_manual(values = CATEGORIELLES[1:4], name = "Modèle") +
  scale_x_continuous(labels = axe_pct) +
  labs(title = "Modèle d'assurance selon le canton", x = "Part des assuré·es", y = NULL)

**Exercice 3. Subsides par canton** *(≈ 7 min, facultatif)*

Dans chaque canton, quelle part des assuré·es reçoit un subside pour payer ses primes ?

Pour comparer (OFSP, 2024, tous âges) : Genève 43,9 %, Vaud 36,8 %, Jura 28,5 %, Fribourg 27,9 %, Berne 26,6 %, Valais 23,7 %, Neuchâtel 22,3 %.

In [ ]:
## Exercice 3 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# taux_subside <- assures_ok %>%
#   group_by(___) %>%
#   summarise(pct_subside = 100 * mean(subside)) %>%
#   drop_na()
# taux_subside %>% arrange(desc(pct_subside))

<details><summary><b>Indice</b></summary>

Grouper par la colonne qui contient le canton de domicile. La variable `subside` vaut 1 pour les bénéficiaires et 0 pour les autres : sa moyenne est une proportion.

</details>

**Questions (exercice 3)**
1. Le classement des cantons ressemble-t-il à celui de la statistique fédérale ?
2. Ces écarts entre cantons viennent-ils des revenus, des règles cantonales, ou des deux ?

**Questions**
1. Comment les assuré·es se répartissent-ils selon l'âge et le sexe ? Que se passe-t-il au-delà de 80 ans ?
2. Comment le choix de la franchise évolue-t-il avec l'âge ? Pourquoi les personnes âgées choisissent-elles presque toutes la franchise minimale ? Quelle particularité présentent les 19 à 25 ans ?
3. Les modèles d'assurance diffèrent-ils d'un canton à l'autre ? Quelle hypothèse pour le Valais ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 3. Qui coûte, et pourquoi ?

Le coût est défini comme le montant décompté à la charge de l'assurance obligatoire des soins en 2024, prestations et médicaments confondus. Pour les séjours hospitaliers, seule la part payée par l'assureur (45 %) figure dans les données. La part cantonale (55 %) n'y est pas. Les médicaments, dont la cause n'est pas indiquée, sont rattachés à la maladie.

In [ ]:
# Coût par personne et par cause
par_cause <- prest_ok %>%
  group_by(id_assure, cause) %>%
  summarise(cout = sum(cout_chf), .groups = "drop") %>%
  pivot_wider(names_from = cause, values_from = cout)
medicaments_personne <- med_ok %>% group_by(id_assure) %>% summarise(medicaments = sum(cout_chf))

couts_cause <- assures_ok %>%
  select(id_assure, sexe, age, classe_ofsp, couverture_accident) %>%
  left_join(par_cause, by = "id_assure") %>%
  left_join(medicaments_personne, by = "id_assure") %>%
  mutate(across(c(maladie, maternite, accident, medicaments), ~ replace_na(.x, 0)),
         maladie = maladie + medicaments,
         total = maladie + maternite + accident) %>%
  select(-medicaments)

total_general <- sum(couts_cause$total)
tibble(cause = c("maladie", "maternite", "accident", "total"),
       part_des_couts = pct(c(sum(couts_cause$maladie), sum(couts_cause$maternite), sum(couts_cause$accident),
                              total_general) / total_general, 1))

### 3.1 Dépenses selon l'âge et le sexe

In [ ]:
moyennes_sexe <- couts_cause %>%
  filter(!is.na(classe_ofsp), !is.na(sexe)) %>%
  group_by(classe_ofsp, sexe) %>%
  summarise(total = mean(total), .groups = "drop")

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(moyennes_sexe, aes(classe_ofsp, total, color = sexe, group = sexe)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2.5) +
  scale_color_manual(values = COULEURS_SEXE, labels = NOMS_SEXE, name = NULL) +
  scale_y_continuous(labels = axe_chf) +
  labs(title = "Coût moyen par personne selon l'âge et le sexe", x = "Classe d'âge", y = "CHF")

### 3.2 Pourquoi les femmes de 20 à 45 ans coûtent-elles davantage ?

La variable `cause` distingue les prestations liées à la **maladie**, à la **maternité** (suivi de grossesse, accouchement, suites de couches) et aux **accidents**.

In [ ]:
par_cause_age <- couts_cause %>%
  filter(!is.na(classe_ofsp), !is.na(sexe)) %>%
  group_by(sexe, classe_ofsp) %>%
  summarise(across(c(maladie, maternite, accident), mean), .groups = "drop") %>%
  pivot_longer(c(maladie, maternite, accident), names_to = "cause", values_to = "cout") %>%
  mutate(cause = factor(cause, levels = c("maladie", "maternite", "accident")))

options(repr.plot.width = 11, repr.plot.height = 4)
ggplot(par_cause_age, aes(classe_ofsp, cout, fill = cause)) +
  geom_col(width = 0.8, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  facet_wrap(~ sexe, labeller = as_labeller(NOMS_SEXE)) +
  scale_fill_manual(values = COULEURS_CAUSE, labels = c("Maladie", "Maternité", "Accident"), name = NULL) +
  scale_y_continuous(labels = axe_chf) +
  labs(title = "Coût moyen par personne selon la cause", x = "Classe d'âge", y = "CHF par personne")

In [ ]:
# Écart entre femmes et hommes de 26 à 35 ans, et part expliquée par la maternité
jeunes <- couts_cause %>%
  filter(between(age, 26, 35), !is.na(sexe)) %>%
  group_by(sexe) %>%
  summarise(across(c(maladie, maternite, accident, total), mean))
femmes <- filter(jeunes, sexe == "F")
hommes <- filter(jeunes, sexe == "M")
ecart <- femmes$total - hommes$total
cat(sprintf("Coût moyen à 26-35 ans : femmes CHF %s, hommes CHF %s\n", chf(femmes$total), chf(hommes$total)))
cat(sprintf("Écart : CHF %s, dont maternité : CHF %s (%s de l'écart)\n", chf(ecart), chf(femmes$maternite),
            pct(femmes$maternite / ecart)))
jeunes

**Exercice 4. La part payée par les assuré·es** *(≈ 8 min)*

Hormis la maternité, les prestations donnent lieu à une participation des assuré·es (franchise, quote-part de 10 %, contribution aux frais de séjour hospitalier) : c'est la colonne `participation_chf`. Selon la statistique de l'OFSP (2024, adultes de 26 ans et plus, calcul à partir des tableaux T 2.23 et T 2.24), cette participation représente 7 à 8 % des coûts avec une franchise de 300 CHF et 41 à 49 % avec une franchise de 2 500 CHF, selon le modèle d'assurance. Calculer cette part (en %) selon la franchise.

In [ ]:
## Exercice 4 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# lignes <- bind_rows(prest_ok %>% select(id_assure, cout_chf, participation_chf),
#                     med_ok %>% select(id_assure, cout_chf, participation_chf)) %>%
#   inner_join(assures_ok %>% select(id_assure, franchise_chf), by = "id_assure")
# par_franchise <- lignes %>%
#   group_by(___) %>%
#   summarise(cout_chf = sum(cout_chf), participation_chf = sum(participation_chf)) %>%
#   mutate(pct_paye = 100 * ___ / ___)
# par_franchise

<details><summary><b>Indice</b></summary>

Grouper par la colonne de la franchise. La part payée rapporte la somme des participations à la somme des coûts : quelle colonne au numérateur, laquelle au dénominateur ?

</details>

**Questions (exercice 4)**
1. Quelle part de leurs coûts les assuré·es paient-ils eux-mêmes avec une franchise de 300 CHF ? Et de 2 500 CHF ?
2. Pourquoi cette part est-elle probablement sous-estimée dans les données de l'assureur ?

### 3.3 Les accidents

L'assurance obligatoire des soins ne couvre l'accident que pour les personnes qui ne sont pas assurées contre les accidents par leur employeur (LAA) : les salarié·es travaillant au moins 8 heures par semaine sont couvert·es par la LAA, et leurs accidents n'apparaissent pas dans ces données. La variable `couverture_accident` indique les personnes couvertes par la LAMal.

In [ ]:
accidents <- couts_cause %>%
  filter(!is.na(classe_ofsp)) %>%
  group_by(classe_ofsp) %>%
  summarise(couverture = mean(couverture_accident), cout_accident = mean(accident))

options(repr.plot.width = 7, repr.plot.height = 3.5)
ggplot(accidents, aes(classe_ofsp, couverture)) +
  geom_col(fill = "#2A78D6", width = 0.75) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Couvert·es contre les accidents par la LAMal", x = "Classe d'âge", y = "Part des assuré·es")
ggplot(accidents, aes(classe_ofsp, cout_accident)) +
  geom_col(fill = COULEURS_CAUSE[["accident"]], width = 0.75) +
  scale_y_continuous(labels = axe_chf) +
  labs(title = "Coût moyen des accidents par personne", x = "Classe d'âge", y = "CHF")

**Exercice 5. Accidents des personnes âgées** *(≈ 5 min, facultatif)*

Parmi les personnes de 76 ans et plus couvertes contre les accidents par la LAMal, comparer le coût moyen des accidents des femmes et des hommes.

In [ ]:
## Exercice 5 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# ages_couverts <- couts_cause %>% filter(age >= 76, ___ == 1)
# ages_couverts %>%
#   group_by(___) %>%
#   summarise(personnes = n(), cout_moyen_accidents = mean(accident)) %>%
#   drop_na()

<details><summary><b>Indice</b></summary>

`names(couts_cause)` affiche les colonnes disponibles. La couverture contre les accidents vaut 1 pour les personnes couvertes par la LAMal. Grouper ensuite par sexe.

</details>

**Questions (exercice 5)**
1. Le coût moyen des accidents diffère-t-il entre femmes et hommes âgés ? Quelles explications proposer ?
2. Sur combien de personnes accidentées ce résultat repose-t-il ? (`sum(ages_couverts$accident > 0)`)

**Questions**
1. À quels âges les dépenses des femmes dépassent-elles celles des hommes ? Quelle part de l'écart la maternité explique-t-elle à 26-35 ans, et quelles hypothèses avancer pour le reste ?
2. Les prestations de maternité ne donnent lieu à aucune participation (franchise, quote-part). Quelle est la justification de cette règle ?
3. Pourquoi les accidents apparaissent-ils surtout après 65 ans dans ces données ?
4. Pour comparer des personnes atteintes d'une maladie chronique aux autres assuré·es, faut-il exclure les prestations de cause « accident » ? Et celles de cause « maternité » ? Justifier.

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 3.4 Concentration des dépenses

La courbe suivante classe les assuré·es du plus coûteux au moins coûteux et cumule leurs dépenses.

In [ ]:
tri <- sort(couts_cause$total, decreasing = TRUE)
n_assures <- length(tri)
courbe <- tibble(part_personnes = seq_len(n_assures) / n_assures, cumul = cumsum(tri) / sum(tri))
reperes <- tibble(p = c(0.01, 0.10, 0.20, 0.50)) %>%
  mutate(valeur = courbe$cumul[floor(p * n_assures)],
         etiquette = sprintf("%s des assuré·es : %s des coûts", pct(p), pct(valeur)))

options(repr.plot.width = 6.5, repr.plot.height = 4.5)
ggplot(courbe, aes(part_personnes, cumul)) +
  geom_line(color = "#2A78D6", linewidth = 1) +
  geom_abline(slope = 1, intercept = 0, linetype = "dotted", color = "#7F8C8D") +
  geom_point(data = reperes, aes(p, valeur), color = "#2A78D6", size = 2.5) +
  geom_text(data = reperes, aes(p, valeur, label = etiquette), hjust = -0.08, vjust = 1.5, size = 3.3) +
  scale_x_continuous(labels = axe_pct) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Concentration des dépenses", x = "Assuré·es, du plus au moins coûteux", y = "Part cumulée des dépenses")

In [ ]:
part_des_plus_couteux <- function(depenses, p) {
  # Part des dépenses totales portée par la fraction p des personnes les plus coûteuses
  valeurs <- sort(depenses, decreasing = TRUE)
  sum(valeurs[seq_len(round(p * length(valeurs)))]) / sum(valeurs)
}

concentration <- tibble(
  groupe = c("1 % les plus coûteux", "5 % les plus coûteux", "10 % les plus coûteux", "20 % les plus coûteux",
             "50 % les moins coûteux"),
  part_des_depenses = c(map_dbl(c(0.01, 0.05, 0.10, 0.20), ~ part_des_plus_couteux(couts_cause$total, .x)),
                        1 - part_des_plus_couteux(couts_cause$total, 0.50))) %>%
  mutate(part_des_depenses = pct(part_des_depenses, 1))
cat(sprintf("Assuré·es sans aucun décompte en 2024 : %s\n", pct(mean(couts_cause$total <= 0), 1)))
concentration

In [ ]:
# Qui sont les 10 % les plus coûteux ?
seuil <- quantile(couts_cause$total, 0.9)
hospitalises <- contacts %>% filter(type_prestation == "stationnaire") %>% pull(id_assure) %>% unique()
deces <- assures_ok %>% filter(motif_fin_couverture == "deces") %>% pull(id_assure)
substances <- med_ok %>% group_by(id_assure) %>% summarise(substances = n_distinct(code_atc))

profil_couts <- couts_cause %>%
  left_join(substances, by = "id_assure") %>%
  mutate(groupe = if_else(total >= seuil, "10 % les plus coûteux", "90 % restants"),
         hospitalise = id_assure %in% hospitalises,
         decede = id_assure %in% deces,
         substances = replace_na(substances, 0L))
profil_couts %>%
  group_by(groupe) %>%
  summarise(personnes = n(), age_moyen = mean(age, na.rm = TRUE), pct_femmes = 100 * mean(sexe %in% "F"),
            pct_hospitalises = 100 * mean(hospitalise), pct_decedes = 100 * mean(decede),
            substances_differentes = mean(substances), cout_moyen = mean(total))

**Exercice 6. Concentration chez les jeunes et chez les âgé·es** *(≈ 8 min, facultatif)*

La fonction `part_des_plus_couteux(depenses, p)`, définie plus haut, calcule la part des dépenses portée par la fraction `p` des personnes les plus coûteuses. Comparer la part des dépenses portée par les 10 % les plus coûteux chez les 19 à 35 ans et chez les 66 ans et plus.

In [ ]:
## Exercice 6 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# depenses_jeunes <- couts_cause %>% filter(age < 36) %>% pull(total)
# depenses_agees <- couts_cause %>% filter(age >= ___) %>% pull(total)
# cat(sprintf("19-35 ans : les 10 %% les plus coûteux portent %s des dépenses\n", pct(part_des_plus_couteux(depenses_jeunes, ___))))
# cat(sprintf("66 ans et plus : les 10 %% les plus coûteux portent %s des dépenses\n", pct(part_des_plus_couteux(___, 0.10))))

<details><summary><b>Indice</b></summary>

Le second groupe commence à 66 ans. Une fraction de 10 % s'écrit `0.10`. Le second appel de la fonction porte sur les dépenses du second groupe.

</details>

**Questions (exercice 6)**
1. Dans quel groupe d'âge les dépenses sont-elles le plus concentrées ? Pourquoi ?

**Questions**
1. Les dépenses suivent-elles la « règle des 80/20 » ? Quelle part des dépenses représentent les 1 % les plus coûteux, et la moitié la moins coûteuse des assuré·es ?
2. Pourquoi une part importante des assuré·es n'a-t-elle aucun décompte dans l'année ?
3. Qu'est-ce qui distingue les 10 % les plus coûteux ? Quelles conséquences ce degré de concentration a-t-il pour le financement de l'assurance obligatoire (solidarité, compensation des risques entre assureurs) ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 3.5 Comparer des coûts : deux pièges d'interprétation

Avec des données de remboursement, on est vite tenté de comparer des coûts entre groupes d'assuré·es ou entre prestataires. Les deux pièges suivants montrent chacun un résultat qui semble parler de lui-même.

**Piège d'interprétation A. La franchise élevée fait-elle baisser les coûts ?** *(≈ 8 min)*

La franchise est souvent présentée comme un levier de maîtrise des coûts. Le graphique compare, selon la franchise choisie, le coût moyen par personne et la part des personnes sans aucun décompte.

In [ ]:
# Coût moyen et part des personnes sans aucun décompte, selon la franchise
selon_franchise <- couts_cause %>%
  left_join(assures_ok %>% select(id_assure, franchise_chf), by = "id_assure") %>%
  group_by(franchise_chf) %>%
  summarise(cout_moyen = mean(total), sans_decompte = mean(total <= 0))

options(repr.plot.width = 6, repr.plot.height = 3.5)
ggplot(selon_franchise, aes(factor(franchise_chf), cout_moyen)) +
  geom_col(fill = "#2A78D6", width = 0.7) +
  scale_y_continuous(labels = axe_chf) +
  labs(title = "Coût moyen par personne", x = "Franchise (CHF)", y = "CHF")
ggplot(selon_franchise, aes(factor(franchise_chf), sans_decompte)) +
  geom_col(fill = "#7F8C8D", width = 0.7) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Personnes sans aucun décompte", x = "Franchise (CHF)", y = "Part des assuré·es")

> **Conclusion tentante.** « Avec une franchise de 2 500 CHF, les assuré·es coûtent cinq fois moins qu'avec une franchise de 300 CHF, et un tiers d'entre eux n'ont eu besoin d'aucun soin. Relever la franchise minimale réduirait donc fortement les dépenses de l'assurance obligatoire. »

**Question (piège A).** Pourquoi les données de remboursement ne permettent-elles pas de tirer cette conclusion ? Distinguer ce qui tient au choix des assuré·es, au comportement induit par la franchise et à la manière dont les données sont produites. *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification : qui choisit quelle franchise ?
profil_couts %>%
  left_join(assures_ok %>% select(id_assure, franchise_chf), by = "id_assure") %>%
  group_by(franchise_chf) %>%
  summarise(personnes = n(), age_moyen = mean(age, na.rm = TRUE),
            pct_hospitalises = 100 * mean(hospitalise), substances_differentes = mean(substances))

**Question (piège A, suite).** Que montre ce tableau ? Pourquoi la part des personnes hospitalisées renseigne-t-elle mieux sur l'état de santé que le nombre de consultations ? Le même raisonnement s'applique-t-il aux modèles d'assurance alternatifs (médecin de famille, HMO, télémédecine), dont les assuré·es coûtent eux aussi beaucoup moins que ceux du modèle standard ?

**Piège d'interprétation B. Peut-on repérer les médecins dépensiers ?** *(≈ 8 min)*

Les données indiquent le prestataire de chaque prestation. Chaque assuré·e est rattaché·e au cabinet de médecine générale qu'il ou elle a consulté le plus souvent en 2024. Le graphique compare le coût moyen des patient·es (prestations et médicaments) des cabinets qui en comptent au moins huit dans les données.

In [ ]:
# Rattachement de chaque assuré·e au cabinet de médecine générale le plus consulté en 2024
cabinet <- prest_ok %>%
  filter(type_prestation == "medecin_generaliste", cout_chf > 0) %>%
  group_by(id_assure, id_prestataire) %>%
  summarise(jours = n_distinct(date_prestation), .groups = "drop") %>%
  arrange(id_assure, id_prestataire) %>%
  group_by(id_assure) %>%
  slice_max(jours, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  select(id_assure, cabinet = id_prestataire)
patients <- profil_couts %>% inner_join(cabinet, by = "id_assure")

# Coût moyen par patient·e des cabinets comptant au moins 8 patient·es
cabinets <- patients %>%
  group_by(cabinet) %>%
  summarise(patients = n(), cout_moyen = mean(total)) %>%
  filter(patients >= 8) %>%
  arrange(cout_moyen)
graphique <- bind_rows(head(cabinets, 10) %>% mutate(groupe = "10 cabinets les moins coûteux"),
                       tail(cabinets, 10) %>% mutate(groupe = "10 cabinets les plus coûteux"))

options(repr.plot.width = 8, repr.plot.height = 5.5)
ggplot(graphique, aes(cout_moyen, fct_reorder(cabinet, cout_moyen), fill = groupe)) +
  geom_col(width = 0.7) +
  geom_vline(xintercept = mean(patients$total), linetype = "dashed") +
  scale_fill_manual(values = c("#2A78D6", "#EB6834"), name = NULL) +
  scale_x_continuous(labels = axe_chf) +
  labs(title = "Coût moyen par patient·e, selon le cabinet de médecine générale",
       subtitle = "Ligne pointillée : moyenne de l'ensemble des patient·es", x = "CHF par patient·e", y = NULL) +
  theme(legend.position = "bottom")

> **Conclusion tentante.** « Les patient·es des cabinets les plus coûteux coûtent jusqu'à trente fois plus que ceux des cabinets les moins coûteux. L'assureur dispose ainsi de la liste des médecins dépensiers, auxquels il peut demander de justifier leurs coûts. »

**Question (piège B).** Les données permettent-elles d'identifier les cabinets coûteux ? Permettent-elles d'expliquer pourquoi ils le sont ? Proposer des explications autres que la pratique des médecins. *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification : patientèle des 20 cabinets du graphique
cat(sprintf("%s personnes rattachées à %s cabinets, soit %.1f patient·es par cabinet en moyenne\n",
            chf(nrow(cabinet)), chf(n_distinct(cabinet$cabinet)), nrow(cabinet) / n_distinct(cabinet$cabinet)))
patients %>%
  filter(cabinet %in% graphique$cabinet) %>%
  mutate(groupe = if_else(cabinet %in% tail(cabinets, 10)$cabinet, "10 plus coûteux", "10 moins coûteux")) %>%
  group_by(groupe, cabinet) %>%
  summarise(patients = n(), age_moyen = mean(age, na.rm = TRUE), hospitalises = sum(hospitalise),
            pct_patient_le_plus_couteux = 100 * max(total) / sum(total), .groups = "drop_last") %>%
  summarise(across(c(patients, age_moyen, hospitalises, pct_patient_le_plus_couteux), mean))

**Question (piège B, suite).** Que montre ce tableau ? Combien de patient·es par cabinet un assureur de 15 000 assuré·es observe-t-il ? De quelles données et de quelles méthodes faudrait-il disposer pour comparer équitablement des médecins ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 4. Personnes traitées pour hypertension

Les sections précédentes ont montré que les dépenses augmentent avec l'âge et se concentrent sur une minorité de personnes, souvent atteintes de maladies chroniques. Cette section étudie l'une d'elles, l'hypertension artérielle. Comme discuté en section 3, le recours aux soins et les coûts sont calculés sur les prestations de cause « maladie » et sur les médicaments : les accidents et la maternité, dont l'observation dépend de la couverture et du sexe, en sont exclus.

### 4.1 Identification des personnes traitées

Les données de facturation ambulatoire ne comportent pas de diagnostic. Les personnes traitées sont donc identifiées à partir des médicaments délivrés, selon la définition opérationnelle suivante : **au moins deux délivrances en 2024 d'un antihypertenseur**, c'est-à-dire d'un médicament dont le code ATC commence par l'un des préfixes ci-dessous. Les personnes ainsi identifiées forment le groupe « HTA traitée » (hypertension artérielle traitée).

| C02 | C03 | C07 | C08 | C09 |
|---|---|---|---|---|
| action centrale / alpha-bloquants | diurétiques | bêtabloquants | inhibiteurs calciques | IEC / sartans (ARA) |

In [ ]:
# Paramètres de la définition opérationnelle (modifiables, l'exercice 7 teste le seuil)
CLASSES_ANTIHYPERTENSEURS <- c("C02", "C03", "C07", "C08", "C09")
SEUIL_DELIVRANCES <- 2

In [ ]:
antihta <- med_ok %>% filter(str_sub(code_atc, 1, 3) %in% CLASSES_ANTIHYPERTENSEURS)
nb_delivrances <- antihta %>% count(id_assure, name = "delivrances")
ids_hta <- nb_delivrances %>% filter(delivrances >= SEUIL_DELIVRANCES) %>% pull(id_assure)

assures_ok <- assures_ok %>%
  mutate(groupe = factor(if_else(id_assure %in% ids_hta, "HTA traitée", "Autres assuré·es"), levels = GROUPES))
hta <- assures_ok$groupe == "HTA traitée"

cat(sprintf("%s personnes traitées pour hypertension, soit %s des assuré·es\n", chf(sum(hta)), pct(mean(hta), 1)))

In [ ]:
distribution <- nb_delivrances %>% count(delivrances = pmin(delivrances, 20))

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(distribution, aes(delivrances, n)) +
  geom_col(fill = "#34495E", width = 0.8) +
  geom_vline(aes(xintercept = SEUIL_DELIVRANCES - 0.5,
                 linetype = sprintf("Seuil retenu : %d délivrances", SEUIL_DELIVRANCES)), color = "#C0392B") +
  scale_linetype_manual(values = "dashed", name = NULL) +
  scale_x_continuous(breaks = 1:20) +
  labs(title = "Nombre de délivrances d'antihypertenseurs par personne (20 = 20 ou plus)",
       x = "Délivrances en 2024", y = "Nombre de personnes") +
  theme(legend.position = "top")

Les bêtabloquants et les diurétiques ont d'autres indications que l'hypertension. Le tableau suivant décrit les personnes identifiées sur la base d'une seule classe thérapeutique, bêtabloquant (C07) ou diurétique de l'anse (C03C).

In [ ]:
classes_par_personne <- antihta %>%
  group_by(id_assure) %>%
  summarise(classes = list(sort(unique(str_sub(code_atc, 1, 3)))))
une_seule_classe <- classes_par_personne %>%
  filter(lengths(classes) == 1, id_assure %in% ids_hta) %>%
  pull(id_assure)
anticoagules <- med_ok %>% filter(str_starts(code_atc, "B01AF")) %>% pull(id_assure) %>% unique()

antihta %>%
  filter(id_assure %in% une_seule_classe, str_detect(code_atc, "^(C07|C03C)")) %>%
  distinct(id_assure, code_atc) %>%
  inner_join(referentiel %>% select(code_atc, substance), by = "code_atc") %>%
  inner_join(assures_ok %>% select(id_assure, age, sexe), by = "id_assure") %>%
  mutate(aussi_anticoagulant = id_assure %in% anticoagules) %>%
  group_by(substance) %>%
  summarise(personnes = n_distinct(id_assure), age_median = median(age, na.rm = TRUE),
            pct_femmes = 100 * mean(sexe %in% "F"), pct_avec_anticoagulant = 100 * mean(aussi_anticoagulant)) %>%
  arrange(desc(personnes))

**Exercice 7. Sensibilité de la définition** *(≈ 6 min)*

Compter les personnes identifiées avec un seuil de 1, 2, 4 ou 6 délivrances d'antihypertenseurs dans l'année.

In [ ]:
## Exercice 7 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# for (seuil in c(1, 2, 4, 6)) {
#   n <- sum(nb_delivrances$delivrances >= ___)
#   cat(sprintf("Au moins %d délivrance(s) : %s personnes (%s des assuré·es)\n", seuil, chf(n), pct(n / nrow(assures_ok), 1)))
# }

<details><summary><b>Indice</b></summary>

À chaque passage de la boucle, la variable `seuil` prend l'une des valeurs de la liste. La colonne `nb_delivrances$delivrances` contient le nombre de délivrances de chaque personne.

</details>

**Questions (exercice 7)**
1. Comment la proportion de personnes identifiées varie-t-elle avec le seuil ?
2. Les traitements au long cours sont souvent délivrés en boîtes de 98 ou 100 comprimés. Combien de délivrances par an une personne traitée toute l'année par un seul médicament reçoit-elle alors ? Quelle conséquence pour le choix du seuil ?

**Questions**
1. La proportion de personnes identifiées est-elle plausible ?
2. Quelles personnes cette définition inclut-elle à tort ? On pourra considérer l'âge et le sexe des personnes traitées par propranolol, ainsi que l'association d'un bêtabloquant et d'un anticoagulant.

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.2 Caractéristiques des personnes traitées

Les autres maladies chroniques sont, elles aussi, approchées par les médicaments délivrés : antidiabétiques (A10), statines (C10) et anticoagulants oraux directs (B01AF).

In [ ]:
prevalence <- assures_ok %>%
  filter(!is.na(classe_age), !is.na(sexe)) %>%
  group_by(classe_age, sexe) %>%
  summarise(part = mean(groupe == "HTA traitée"), .groups = "drop")

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(prevalence, aes(classe_age, part, fill = sexe)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.75) +
  scale_fill_manual(values = COULEURS_SEXE, labels = NOMS_SEXE, name = NULL) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Part des personnes traitées pour hypertension", x = "Classe d'âge", y = NULL)

**Piège d'interprétation C. Que mesure l'indicateur « HTA traitée » ?** *(≈ 10 min)*

Le graphique compare la part des personnes identifiées comme traitées pour hypertension dans les données à la part des personnes qui déclarent une hypertension dans l'Enquête suisse sur la santé (ESS 2022, Obsan, où la première classe d'âge va de 15 à 24 ans).

In [ ]:
# Hypertension traitée (données) et hypertension déclarée (Enquête suisse sur la santé 2022)
CLASSES_ESS <- c("19-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+")
ESS_HTA <- tibble(classe = factor(CLASSES_ESS, levels = CLASSES_ESS),
                  F = c(0.7, 1.7, 3.2, 8.8, 21.0, 35.7, 54.9) / 100,
                  M = c(2.4, 3.8, 7.6, 18.0, 34.2, 47.6, 59.4) / 100) %>%
  pivot_longer(c(F, M), names_to = "sexe", values_to = "ess")
hta_donnees <- assures_ok %>%
  mutate(classe = cut(age, c(18, 24, 34, 44, 54, 64, 74, 120), labels = CLASSES_ESS)) %>%
  filter(!is.na(classe), !is.na(sexe)) %>%
  group_by(classe, sexe) %>%
  summarise(donnees = mean(groupe == "HTA traitée"), .groups = "drop") %>%
  left_join(ESS_HTA, by = c("classe", "sexe"))

options(repr.plot.width = 11, repr.plot.height = 4)
ggplot(hta_donnees, aes(classe)) +
  geom_col(aes(y = donnees, fill = sexe), width = 0.7, show.legend = FALSE) +
  geom_point(aes(y = ess, shape = "ESS 2022 : hypertension déclarée"), size = 3) +
  facet_wrap(~ sexe, labeller = as_labeller(NOMS_SEXE)) +
  scale_fill_manual(values = COULEURS_SEXE) +
  scale_shape_manual(values = 18, name = NULL) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Hypertension traitée et hypertension déclarée",
       subtitle = "Barres : données, personnes traitées (au moins 2 délivrances)", x = "Classe d'âge", y = "Part des personnes") +
  theme(legend.position = "top")
cat("Rapport entre les données et l'enquête :\n")
hta_donnees %>%
  mutate(rapport = round(donnees / ess, 1)) %>%
  select(sexe, classe, rapport) %>%
  pivot_wider(names_from = classe, values_from = rapport)

> **Conclusion tentante.** « Chez les femmes de moins de 45 ans, les données comptent nettement plus de personnes traitées pour hypertension que l'enquête ne compte de personnes hypertendues (trois fois plus entre 19 et 24 ans), alors que chez les hommes les deux sources concordent : l'hypertension des jeunes femmes est sous-estimée par les enquêtes. Après 55 ans, les deux sources concordent : environ une personne sur deux est hypertendue à partir de 75 ans, et le groupe « Autres assuré·es » rassemble les personnes qui ne le sont pas. »

**Question (piège C).** Pourquoi les données de remboursement ne permettent-elles pas de tirer ces conclusions ? Que mesure exactement l'indicateur « HTA traitée » ? *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification 1 : sans les personnes identifiées uniquement par un bêtabloquant (C07), dont l'indication est inconnue
bb_seul <- classes_par_personne %>%
  filter(map_lgl(classes, ~ identical(.x, "C07")), id_assure %in% ids_hta) %>%
  pull(id_assure)
cat("Femmes, en % :\n")
assures_ok %>%
  filter(sexe == "F") %>%
  mutate(classe = cut(age, c(18, 24, 34, 44, 54, 64, 74, 120), labels = CLASSES_ESS)) %>%
  filter(!is.na(classe)) %>%
  group_by(classe) %>%
  summarise(donnees = 100 * mean(groupe == "HTA traitée"),
            donnees_sans_betabloquant_seul = 100 * mean(groupe == "HTA traitée" & !(id_assure %in% bb_seul))) %>%
  left_join(ESS_HTA %>% filter(sexe == "F") %>% transmute(classe, ess_2022 = 100 * ess), by = "classe") %>%
  mutate(across(where(is.numeric), ~ round(.x, 1)))

# Vérification 2 : selon l'OMS (2023), 56 % seulement des personnes hypertendues de 30 à 79 ans sont traitées en Suisse
p <- assures_ok %>% filter(between(age, 30, 79)) %>% summarise(mean(groupe == "HTA traitée")) %>% pull()
cat(sprintf(paste("30-79 ans : %s de personnes traitées. Si 56 %% des hypertendu·es sont traité·es, la prévalence serait d'environ %s,",
                  "et environ %s des « autres assuré·es » seraient hypertendu·es sans traitement.\n"),
            pct(p), pct(p / 0.56), pct((p / 0.56 - p) / (1 - p))))

**Question (piège C, suite).** Que montrent ces vérifications ? Quelles données faudrait-il pour mesurer la prévalence de l'hypertension ? Quelle conséquence la présence de personnes hypertendues non traitées dans le groupe « Autres assuré·es » a-t-elle sur les comparaisons de cette section et des suivantes ?

In [ ]:
# Indicateurs de comorbidité fondés sur les délivrances
PROXYS <- c("Diabète (A10)" = "A10", "Statine (C10)" = "C10", "Anticoagulant (B01AF)" = "B01AF")
for (nom in names(PROXYS)) {
  ids <- med_ok %>% filter(str_starts(code_atc, PROXYS[[nom]])) %>% pull(id_assure)
  assures_ok[[nom]] <- assures_ok$id_assure %in% ids
}

profil <- assures_ok %>%
  group_by(groupe) %>%
  summarise(personnes = n(), age_moyen = mean(age, na.rm = TRUE), pct_femmes = 100 * mean(sexe %in% "F"),
            pct_subside = 100 * mean(subside),
            across(all_of(names(PROXYS)), ~ 100 * mean(.x), .names = "% {.col}"))
profil

In [ ]:
comorbidites <- assures_ok %>%
  group_by(groupe) %>%
  summarise(across(all_of(names(PROXYS)), mean)) %>%
  pivot_longer(-groupe, names_to = "traitement", values_to = "part") %>%
  mutate(traitement = factor(traitement, levels = rev(names(PROXYS))))

options(repr.plot.width = 8, repr.plot.height = 3.5)
ggplot(comorbidites, aes(part, traitement, fill = groupe)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.75) +
  scale_fill_manual(values = COULEURS, name = NULL, guide = guide_legend(reverse = TRUE)) +
  scale_x_continuous(labels = axe_pct) +
  labs(title = "Autres traitements chroniques", x = "Part des personnes", y = NULL)

**Exercice 8. Statines à âge égal** *(≈ 7 min, facultatif)*

Les personnes traitées pour hypertension sont plus âgées que les autres assuré·es. Comparer, par classe d'âge, la part (en %) de personnes recevant une statine dans les deux groupes.

In [ ]:
## Exercice 8 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# statines_age <- assures_ok %>%
#   group_by(___, groupe) %>%
#   summarise(pct_statine = 100 * mean(`Statine (C10)`), .groups = "drop") %>%
#   drop_na() %>%
#   pivot_wider(names_from = groupe, values_from = pct_statine)
# statines_age

<details><summary><b>Indice</b></summary>

Le premier critère de regroupement est la classe d'âge : reprendre le nom de la colonne utilisée dans les graphiques précédents, sans guillemets.

</details>

**Questions (exercice 8)**
1. L'écart de traitement par statine entre les deux groupes persiste-t-il à âge égal ?
2. Que suggère ce résultat pour la prise en charge des personnes hypertendues ?

La cellule suivante compare la distribution d'une variable entre les deux groupes. Paramètre modifiable : `VARIABLE` (`franchise_chf`, `modele_assurance`, `canton` ou `subside`).

In [ ]:
VARIABLE <- "franchise_chf"   # paramètre modifiable

donnees <- assures_ok %>% mutate(franchise_chf = GROUPES_FRANCHISE[as.character(franchise_chf)])
repartition <- donnees %>%
  filter(!is.na(.data[[VARIABLE]])) %>%
  count(groupe, valeur = as.character(.data[[VARIABLE]])) %>%
  group_by(groupe) %>% mutate(part = n / sum(n)) %>% ungroup()
niveaux <- if (VARIABLE == "franchise_chf") c("300", "500", "1000-2000", "2500") else sort(unique(repartition$valeur))
couleurs <- if (VARIABLE == "franchise_chf") RAMPE else CATEGORIELLES[seq_along(niveaux)]
repartition <- repartition %>% mutate(valeur = factor(valeur, levels = niveaux))

options(repr.plot.width = 8, repr.plot.height = 2.8)
ggplot(repartition, aes(part, fct_rev(groupe), fill = valeur)) +
  geom_col(width = 0.7, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  scale_fill_manual(values = setNames(couleurs, niveaux), labels = libelle, name = NULL) +
  scale_x_continuous(labels = axe_pct) +
  labs(title = paste(libelle(VARIABLE), "selon le groupe"), x = NULL, y = NULL)

**Questions**
1. Comment caractériser, en deux phrases, les personnes traitées pour hypertension ?
2. Comment expliquer la fréquence plus élevée de la franchise minimale dans ce groupe ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.3 Traitements antihypertenseurs

Certains médicaments sont des associations fixes, qui réunissent plusieurs classes thérapeutiques sous un seul code ATC (par exemple `C09DA` : antagoniste de l'angiotensine II et diurétique). Chaque code est donc converti en la ou les classes qu'il contient.

In [ ]:
CLASSES <- list(   # début du code ATC → classes contenues
  C09AA = "IEC", C09BA = c("IEC", "Diurétique"), C09BB = c("IEC", "Inhibiteur calcique"),
  C09CA = "Sartan (ARA)", C09DA = c("Sartan (ARA)", "Diurétique"),
  C09DB = c("Sartan (ARA)", "Inhibiteur calcique"),
  C09DX = c("Sartan (ARA)", "Inhibiteur calcique", "Diurétique"),
  C08 = "Inhibiteur calcique", C07 = "Bêtabloquant", C02 = "Autre antihypertenseur",
  C03A = "Diurétique", C03B = "Diurétique", C03C = "Diurétique de l'anse", C03D = "Antagoniste aldostérone")

classes_du_code <- function(code_atc) {
  for (debut in names(CLASSES)[order(-nchar(names(CLASSES)))]) {
    if (startsWith(code_atc, debut)) return(CLASSES[[debut]])
  }
  "Non classé"
}

trt <- antihta %>%
  filter(id_assure %in% ids_hta) %>%
  mutate(classe = map(code_atc, classes_du_code), combinaison_fixe = lengths(classe) > 1)
trt_classes <- trt %>% unnest_longer(classe)

par_personne <- trt %>%
  group_by(id_assure) %>%
  summarise(nb_codes = n_distinct(code_atc), combinaison_fixe = any(combinaison_fixe)) %>%
  left_join(trt_classes %>% group_by(id_assure) %>% summarise(nb_classes = n_distinct(classe)), by = "id_assure")
cat(sprintf("%s des personnes reçoivent au moins une combinaison fixe\n", pct(mean(par_personne$combinaison_fixe))))

In [ ]:
part_classes <- trt_classes %>%
  group_by(classe) %>%
  summarise(part = n_distinct(id_assure) / nrow(par_personne))

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(part_classes, aes(part, fct_reorder(classe, part))) +
  geom_col(fill = COULEURS[["HTA traitée"]], width = 0.75) +
  scale_x_continuous(labels = axe_pct) +
  labs(title = "Classes utilisées (au moins une fois en 2024)", x = "Part des personnes traitées", y = NULL)

In [ ]:
comparaison <- bind_rows(
  par_personne %>% count(nombre = pmin(nb_codes, 4)) %>% mutate(mesure = "Compté en codes ATC"),
  par_personne %>% count(nombre = pmin(nb_classes, 4)) %>% mutate(mesure = "Compté en classes")) %>%
  group_by(mesure) %>% mutate(part = n / sum(n)) %>% ungroup() %>%
  mutate(nombre = factor(if_else(nombre == 4, "4+", as.character(nombre)), levels = c("1", "2", "3", "4+")),
         mesure = factor(mesure, levels = c("Compté en codes ATC", "Compté en classes")))

ggplot(comparaison, aes(nombre, part, fill = mesure)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.75) +
  scale_fill_manual(values = c("#BDC3C7", COULEURS[["HTA traitée"]]), name = NULL) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Nombre de médicaments antihypertenseurs par personne", x = "Nombre", y = "Part des personnes")

In [ ]:
polytherapie <- par_personne %>%
  left_join(assures_ok %>% select(id_assure, classe_age), by = "id_assure") %>%
  filter(!is.na(classe_age)) %>%
  mutate(classes = factor(pmin(nb_classes, 3), levels = 1:3, labels = c("1 classe", "2 classes", "3 classes ou +"))) %>%
  count(classe_age, classes) %>%
  group_by(classe_age) %>% mutate(part = n / sum(n)) %>% ungroup()

ggplot(polytherapie, aes(classe_age, part, fill = classes)) +
  geom_col(width = 0.75, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  scale_fill_manual(values = c("#F5B7B1", "#E74C3C", "#922B21"), name = NULL) +
  scale_y_continuous(labels = axe_pct) +
  labs(title = "Nombre de classes selon l'âge", x = "Classe d'âge", y = NULL)

**Questions**
1. Quelles sont les classes les plus utilisées ? La monothérapie est-elle majoritaire, et comment évolue-t-elle avec l'âge ?
2. Pourquoi un dénombrement des codes ATC sous-estime-t-il la fréquence des associations thérapeutiques ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### 4.4 Recours aux soins

Le recours aux soins est mesuré par le nombre de contacts de cause « maladie » (définis à la section 1) par personne et par type de prestation.

In [ ]:
contacts_maladie <- contacts %>% filter(cause == "maladie")
recours <- contacts_maladie %>%
  count(id_assure, type_prestation) %>%
  pivot_wider(names_from = type_prestation, values_from = n, values_fill = 0L)
recours <- assures_ok %>%
  select(id_assure, groupe, classe_age) %>%
  left_join(recours, by = "id_assure") %>%
  mutate(across(-c(id_assure, groupe, classe_age), ~ replace_na(.x, 0L)),
         hospitalise = stationnaire > 0)

TYPES <- c("medecin_generaliste", "medecin_specialiste", "laboratoire", "ambulatoire_hospitalier", "physiotherapie")
moyennes <- recours %>%
  group_by(groupe) %>%
  summarise(across(all_of(TYPES), mean)) %>%
  pivot_longer(-groupe, names_to = "type_prestation", values_to = "contacts")

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(moyennes, aes(contacts, factor(libelle(type_prestation), levels = rev(libelle(TYPES))), fill = groupe)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.75) +
  scale_fill_manual(values = COULEURS, name = NULL, guide = guide_legend(reverse = TRUE)) +
  labs(title = "Nombre moyen de contacts par personne en 2024", x = "Contacts", y = NULL)

In [ ]:
cat("Part des personnes hospitalisées au moins une fois :\n")
recours %>% group_by(groupe) %>% summarise(hospitalisees = pct(mean(hospitalise), 1))

**Piège d'interprétation D. Les associations d'antihypertenseurs sont-elles dangereuses ?** *(≈ 7 min)*

Le graphique compare la part des personnes traitées hospitalisées au moins une fois en 2024, selon le nombre de classes d'antihypertenseurs reçues (section 4.3).

In [ ]:
# Hospitalisation selon le nombre de classes d'antihypertenseurs (personnes traitées)
poly <- par_personne %>%
  left_join(recours %>% select(id_assure, hospitalise), by = "id_assure") %>%
  mutate(traitement = factor(pmin(nb_classes, 3), levels = 1:3, labels = c("1 classe", "2 classes", "3 classes ou +")))
taux_hospitalisation <- poly %>% group_by(traitement) %>% summarise(taux = mean(hospitalise))

options(repr.plot.width = 6, repr.plot.height = 3.8)
ggplot(taux_hospitalisation, aes(traitement, taux)) +
  geom_col(fill = COULEURS[["HTA traitée"]], width = 0.6) +
  geom_text(aes(label = pct(taux)), vjust = -0.5) +
  scale_y_continuous(labels = axe_pct, limits = c(0, 1.2 * max(taux_hospitalisation$taux))) +
  labs(title = "Personnes hospitalisées au moins une fois en 2024", x = "Traitement antihypertenseur",
       y = "Part des personnes traitées")

> **Conclusion tentante.** « Plus les personnes reçoivent de classes d'antihypertenseurs, plus elles sont hospitalisées : 14 % avec une seule classe, 26 % avec trois classes ou plus. Les associations d'antihypertenseurs augmentent le risque d'hospitalisation et devraient être limitées. »

**Question (piège D).** Pourquoi ces données ne permettent-elles pas de conclure que les associations d'antihypertenseurs provoquent des hospitalisations ? Proposer d'autres explications. *Répondre avant d'exécuter la cellule suivante.*

In [ ]:
# Vérification : âge et maladies associées selon le nombre de classes
MARQUEURS <- c("Anticoagulant (B01AF)" = "B01AF", "Diurétique de l'anse (C03C)" = "C03C", "Antidiabétique (A10)" = "A10")
for (nom in names(MARQUEURS)) {
  ids <- med_ok %>% filter(str_starts(code_atc, MARQUEURS[[nom]])) %>% pull(id_assure)
  poly[[nom]] <- poly$id_assure %in% ids
}
poly %>%
  left_join(assures_ok %>% select(id_assure, age), by = "id_assure") %>%
  group_by(traitement) %>%
  summarise(personnes = n(), age_moyen = mean(age, na.rm = TRUE),
            across(all_of(names(MARQUEURS)), ~ 100 * mean(.x), .names = "% {.col}"))

**Question (piège D, suite).** Que montre ce tableau ? Que signifie l'expression « confusion par l'indication » ? Quelles informations, absentes des données de remboursement, et quel type d'étude faudrait-il pour évaluer l'effet des associations d'antihypertenseurs ?

Les personnes traitées sont en moyenne plus âgées. On refait donc la comparaison par classe d'âge, pour séparer l'effet de l'âge de celui de l'hypertension.

In [ ]:
generaliste_age <- recours %>%
  filter(!is.na(classe_age)) %>%
  group_by(classe_age, groupe) %>%
  summarise(contacts = mean(medecin_generaliste), .groups = "drop")

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(generaliste_age, aes(classe_age, contacts, color = groupe, group = groupe)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2.5) +
  scale_color_manual(values = COULEURS, name = NULL) +
  labs(title = "Consultations chez le généraliste, à âge égal", x = "Classe d'âge", y = "Contacts moyens par personne")

In [ ]:
specialites <- contacts_maladie %>%
  filter(type_prestation == "medecin_specialiste", !is.na(specialite)) %>%
  inner_join(assures_ok %>% select(id_assure, groupe), by = "id_assure") %>%
  count(groupe, specialite) %>%
  group_by(groupe) %>% mutate(part = n / sum(n)) %>% ungroup()
ordre <- specialites %>% filter(groupe == "HTA traitée") %>% arrange(part) %>% pull(specialite)

options(repr.plot.width = 8, repr.plot.height = 5)
ggplot(specialites, aes(part, factor(specialite, levels = ordre), fill = groupe)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.75) +
  scale_y_discrete(labels = libelle) +
  scale_fill_manual(values = COULEURS, name = NULL, guide = guide_legend(reverse = TRUE)) +
  scale_x_continuous(labels = axe_pct) +
  labs(title = "Répartition des consultations de spécialistes", x = "Part des consultations", y = NULL)

**Exercice 9. Des personnes traitées sans consultation ?** *(≈ 8 min)*

Un traitement antihypertenseur nécessite une ordonnance médicale. Calculer, selon la franchise, la part (en %) des personnes traitées pour hypertension n'ayant aucun contact de généraliste enregistré dans les données.

In [ ]:
## Exercice 9 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# hta_recours <- recours %>%
#   filter(groupe == "HTA traitée") %>%
#   left_join(assures_ok %>% select(id_assure, franchise_chf), by = "id_assure")
# sans_generaliste <- hta_recours %>%
#   group_by(___) %>%
#   summarise(pct_sans_generaliste = 100 * mean(medecin_generaliste == ___))
# sans_generaliste

<details><summary><b>Indice</b></summary>

Grouper par la colonne de la franchise. Dans chaque groupe, on cherche la part des personnes dont le nombre de contacts de généraliste est nul.

</details>

**Questions (exercice 9)**
1. Quelles raisons médicales peuvent expliquer qu'une personne traitée n'ait aucune consultation de généraliste enregistrée ?
2. Pourquoi cette part augmente-t-elle avec la franchise ? Qu'en déduire pour la mesure du recours aux soins ?

**Questions**
1. Les personnes traitées recourent-elles davantage aux soins, et pour quels types de prestations ?
2. La différence persiste-t-elle à âge égal ? Quels autres facteurs pourraient l'expliquer ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

**Analyse complémentaire : continuité des soins.** La variable `id_prestataire` permet de savoir si une personne consulte toujours le même généraliste.

In [ ]:
continuite <- prest_ok %>%
  filter(type_prestation == "medecin_generaliste", cause == "maladie", cout_chf > 0) %>%
  group_by(id_assure) %>%
  summarise(jours = n_distinct(date_prestation), medecins = n_distinct(id_prestataire)) %>%
  filter(jours >= 3) %>%
  inner_join(assures_ok %>% select(id_assure, groupe), by = "id_assure")
cat("Personnes avec au moins 3 consultations de généraliste n'ayant vu qu'un seul généraliste :\n")
continuite %>% group_by(groupe) %>% summarise(un_seul_generaliste = pct(mean(medecins == 1)))

**Question.** Pourquoi la continuité des soins est-elle un enjeu particulier pour les personnes atteintes d'une maladie chronique ? Quel lien avec les modèles d'assurance « médecin de famille » ?

### 4.5 Coûts

Le coût comprend les prestations de cause « maladie » et les médicaments, décomptés en 2024 à la charge de l'assurance obligatoire des soins.

In [ ]:
par_poste <- prest_ok %>%
  filter(cause == "maladie") %>%
  group_by(id_assure, type_prestation) %>%
  summarise(cout = sum(cout_chf), .groups = "drop") %>%
  pivot_wider(names_from = type_prestation, values_from = cout)
med_antihta <- antihta %>% group_by(id_assure) %>% summarise(med_antihypertenseurs = sum(cout_chf))
med_autres <- med_ok %>% anti_join(antihta, by = "id_ligne") %>% group_by(id_assure) %>% summarise(med_autres = sum(cout_chf))
POSTES <- c(setdiff(names(par_poste), "id_assure"), "med_antihypertenseurs", "med_autres")

couts <- assures_ok %>%                       # toutes les personnes assurées, même sans prestation
  select(id_assure, groupe, classe_age, franchise_chf, motif_fin_couverture) %>%
  left_join(par_poste, by = "id_assure") %>%
  left_join(med_antihta, by = "id_assure") %>%
  left_join(med_autres, by = "id_assure") %>%
  mutate(across(all_of(POSTES), ~ replace_na(.x, 0)))
couts$total <- rowSums(couts[POSTES])

couts %>%
  group_by(groupe) %>%
  summarise(personnes = n(), moyenne = mean(total), mediane = median(total),
            centile_90 = quantile(total, 0.9), max = max(total)) %>%
  mutate(across(-c(groupe, personnes), round))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(couts %>% filter(total > 0), aes(total, color = groupe)) +
  geom_freqpoly(aes(y = after_stat(density)), bins = 40, linewidth = 0.9) +
  scale_x_log10(labels = axe_chf) +
  scale_color_manual(values = COULEURS, name = NULL) +
  labs(title = "Distribution du coût annuel par personne (échelle logarithmique)", x = "CHF", y = NULL)

In [ ]:
NOMS <- c(stationnaire = "Hospitalisation", ambulatoire_hospitalier = "Ambulatoire hospitalier",
          medecin_specialiste = "Spécialiste", medecin_generaliste = "Généraliste", laboratoire = "Laboratoire",
          med_autres = "Médicaments : autres", med_antihypertenseurs = "Médicaments : antihypertenseurs")
ORDRE <- c("Hospitalisation", "Ambulatoire hospitalier", "Spécialiste", "Généraliste", "Laboratoire",
           "Autres prestations", "Médicaments : autres", "Médicaments : antihypertenseurs")   # ordre et couleurs fixes
postes <- couts %>%
  group_by(groupe) %>%
  summarise(across(all_of(POSTES), mean)) %>%
  pivot_longer(-groupe, names_to = "poste", values_to = "cout") %>%
  mutate(poste = factor(coalesce(unname(NOMS[poste]), "Autres prestations"), levels = ORDRE)) %>%
  group_by(groupe, poste) %>%
  summarise(cout = sum(cout), .groups = "drop")

options(repr.plot.width = 7, repr.plot.height = 5)
ggplot(postes, aes(groupe, cout, fill = poste)) +
  geom_col(width = 0.6, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  scale_fill_manual(values = setNames(CATEGORIELLES, ORDRE), name = NULL, guide = guide_legend(reverse = TRUE)) +
  scale_y_continuous(labels = axe_chf) +
  labs(title = "Coût moyen par personne et par poste", x = NULL, y = "CHF")

postes_hta <- postes %>% filter(groupe == "HTA traitée")
part_antihta <- sum(postes_hta$cout[postes_hta$poste == "Médicaments : antihypertenseurs"]) / sum(postes_hta$cout)
cat(sprintf("Antihypertenseurs = %s du coût moyen des personnes traitées\n", pct(part_antihta, 1)))

In [ ]:
cout_age <- couts %>%
  filter(!is.na(classe_age)) %>%
  group_by(classe_age, groupe) %>%
  summarise(cout = mean(total), .groups = "drop")

options(repr.plot.width = 8, repr.plot.height = 4)
ggplot(cout_age, aes(classe_age, cout, color = groupe, group = groupe)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2.5) +
  scale_color_manual(values = COULEURS, name = NULL) +
  scale_y_continuous(labels = axe_chf) +
  labs(title = "Coût moyen par personne, à âge égal", x = "Classe d'âge", y = "CHF")

In [ ]:
poids <- couts %>%
  group_by(groupe) %>%
  summarise(personnes = n(), couts = sum(total)) %>%
  mutate(personnes = personnes / sum(personnes), couts = couts / sum(couts))
poids_long <- poids %>%
  pivot_longer(-groupe, names_to = "mesure", values_to = "part") %>%
  mutate(mesure = factor(mesure, levels = c("couts", "personnes"), labels = c("Part des coûts", "Part des assuré·es")))

options(repr.plot.width = 8, repr.plot.height = 2.5)
ggplot(poids_long, aes(part, mesure, fill = groupe)) +
  geom_col(width = 0.6, color = "white", linewidth = 0.3, position = position_stack(reverse = TRUE)) +
  geom_text(data = poids_long %>% filter(groupe == "HTA traitée"), aes(x = part / 2, label = pct(part)), color = "white") +
  scale_fill_manual(values = COULEURS, name = NULL) +
  scale_x_continuous(labels = axe_pct) +
  labs(title = "Part des personnes traitées dans l'effectif et dans les coûts", x = NULL, y = NULL)

plus_couteux <- couts %>% arrange(desc(total)) %>% slice_head(n = nrow(couts) %/% 10)
part_hta_top10 <- mean(plus_couteux$groupe == "HTA traitée")
cat(sprintf("Personnes traitées pour HTA parmi les 10 %% les plus coûteux : %s (contre %s de l'ensemble des assuré·es)\n",
            pct(part_hta_top10), pct(mean(hta))))

**Exercice 10. Coût à âge égal : standardisation directe** *(≈ 10 min)*

Les personnes traitées pour hypertension sont plus âgées, et le coût augmente avec l'âge. La standardisation directe estime le coût moyen qu'aurait chaque groupe s'il avait la structure d'âge d'une population de référence, ici l'ensemble des assuré·es : les coûts moyens de chaque classe d'âge sont pondérés par la part de cette classe dans la population de référence. Compléter le calcul, puis comparer le rapport des coûts brut et standardisé.

In [ ]:
## Exercice 10 : décommenter les lignes (les sélectionner, puis Ctrl + / ou Cmd + /) et remplacer chaque ___
# structure <- assures_ok %>% filter(!is.na(classe_age)) %>% count(classe_age) %>% mutate(poids = n / sum(n))
# moyennes_age <- couts %>%
#   filter(!is.na(classe_age)) %>%
#   group_by(classe_age, groupe) %>%
#   summarise(cout_moyen = mean(total), .groups = "drop") %>%
#   left_join(structure %>% select(classe_age, poids), by = "classe_age")
# standardise <- moyennes_age %>% group_by(groupe) %>% summarise(cout = sum(cout_moyen * ___))
# brut <- couts %>% group_by(groupe) %>% summarise(cout = mean(total))
# cat(sprintf("Coût moyen brut          : traitées CHF %s, autres CHF %s, rapport %.1f\n",
#             chf(brut$cout[1]), chf(brut$cout[2]), brut$cout[1] / brut$cout[2]))
# cat(sprintf("Coût moyen standardisé   : traitées CHF %s, autres CHF %s, rapport %.1f\n",
#             chf(standardise$cout[1]), chf(standardise$cout[2]), standardise$cout[1] / standardise$cout[2]))

<details><summary><b>Indice</b></summary>

Les poids sont les parts des classes d'âge dans l'ensemble des assuré·es : regarder les colonnes ajoutées par le `left_join()`.

</details>

**Questions (exercice 10)**
1. Le rapport des coûts diminue-t-il une fois l'âge pris en compte ?
2. L'écart restant mesure-t-il le coût de l'hypertension ? Quels autres facteurs l'expliquent ?

**Questions**
1. Compte tenu de la distribution des coûts, quel indicateur (moyenne ou médiane) convient-il de présenter à un·e décideur·e, et dans quel but ?
2. Quelle part des coûts des personnes traitées est attribuable aux antihypertenseurs ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

### Analyses complémentaires

In [ ]:
# Personnes non assurées toute l'année
couts %>%
  mutate(motif = replace_na(motif_fin_couverture, "assuré·e toute l'année")) %>%
  group_by(motif) %>%
  summarise(personnes = n(), moyenne = round(mean(total)), mediane = round(median(total)))

**Question.** Que suggèrent ces résultats concernant les coûts en fin de vie ? Quel biais introduit la comparaison de personnes assurées pendant des durées différentes ?

**Réponses du groupe** *(double-cliquer sur la cellule pour la modifier)*

---
## 5. Synthèse et rédaction de la note

Le tableau suivant rassemble les principaux résultats. Les valeurs dépendent des choix effectués dans les sections précédentes.

In [ ]:
# Synthèse des principaux résultats
valeur <- function(table, colonne, grp) table[[colonne]][table$groupe == grp]
hospit <- recours %>% group_by(groupe) %>% summarise(taux = mean(hospitalise))
cout_groupe <- couts %>% group_by(groupe) %>% summarise(moyenne = mean(total), mediane = median(total))
generaliste <- moyennes %>% filter(type_prestation == "medecin_generaliste")

tibble(
  indicateur = c("Personnes traitées pour hypertension", "Âge moyen (traitées / autres)", "Part de femmes parmi les traitées",
                 "Traitées avec ≥ 2 classes", "Généraliste, contacts/an (traitées / autres)",
                 "Hospitalisées ≥ 1 fois (traitées / autres)", "Coût moyen (traitées / autres)",
                 "Coût médian (traitées / autres)", "Part des coûts totaux portée par les traitées",
                 "Part des traitées parmi les 10 % les plus coûteux", "Part des antihypertenseurs dans leur coût"),
  valeur = c(
    sprintf("%s (%s des assuré·es)", chf(sum(hta)), pct(mean(hta), 1)),
    sprintf("%.0f / %.0f ans", valeur(profil, "age_moyen", "HTA traitée"), valeur(profil, "age_moyen", "Autres assuré·es")),
    sprintf("%.0f%%", valeur(profil, "pct_femmes", "HTA traitée")),
    pct(mean(par_personne$nb_classes >= 2)),
    sprintf("%.1f / %.1f par an", valeur(generaliste, "contacts", "HTA traitée"), valeur(generaliste, "contacts", "Autres assuré·es")),
    sprintf("%s / %s", pct(valeur(hospit, "taux", "HTA traitée")), pct(valeur(hospit, "taux", "Autres assuré·es"))),
    sprintf("%s / %s CHF", chf(valeur(cout_groupe, "moyenne", "HTA traitée")), chf(valeur(cout_groupe, "moyenne", "Autres assuré·es"))),
    sprintf("%s / %s CHF", chf(valeur(cout_groupe, "mediane", "HTA traitée")), chf(valeur(cout_groupe, "mediane", "Autres assuré·es"))),
    pct(valeur(poids, "couts", "HTA traitée")),
    pct(part_hta_top10),
    pct(part_antihta, 1)))

### Structure de la note

**Destinataire :** responsable de la santé publique cantonale.
**Objet :** personnes traitées pour hypertension artérielle : profil, recours aux soins et coûts.

1. **Population concernée.** Effectif, proportion et principales caractéristiques.
2. **Traitements et recours aux soins.** Éléments distinguant les personnes traitées des autres assuré·es.
3. **Coûts associés.** Ordres de grandeur, principaux déterminants et place dans la concentration des dépenses.
4. **Messages principaux.** Deux ou trois, formulés en vue de la décision.
5. **Limite d'interprétation.** La principale limite et ses implications pour la décision (les pièges d'interprétation A à D en donnent des exemples).

Une page, avec au plus un graphique. Des chiffres arrondis, et pas de jargon d'analyse de données.